# Model V1 · Large-scale inference

Run the final freeze cell in notebook 05 first. Review the sample before enabling inference. Outputs are local under `outputs/`. Even masked images can contain missed plates. Keep review material private.

In [ ]:
import hashlib
import json
import math
import platform
from pathlib import Path
from tempfile import TemporaryDirectory
from time import perf_counter

import matplotlib.pyplot as plt
import pandas as pd
import torch
import ultralytics
from PIL import Image
from ultralytics import YOLO

root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "notebooks/05_model_v1_training.ipynb").is_file())
frozen = root / "models/model_v1"
raw_test = root / "data/raw/bdd100k/test"
output = root / "outputs/model_v1_bdd100k_test"
seed = 42
sample_size = 1000
batch_size = 8
padding_fraction = 0.15  # Per side, relative to box width or height
minimum_padding_px = 2
run_inference = True

## 1. Sample the original BDD100K test split

Exclude original image IDs from both project annotation manifests across train, validation and test, including negatives and images later dropped from YOLO. BDD100K filename stems identify source frames. Seeded SHA-256 ranks make the selection independent of directory traversal order.

In [ ]:
def original_id(filename: str) -> str:
    return Path(filename.replace(chr(92), "/")).stem.casefold()

excluded_ids = set()
for dataset in ("poc", "dataset_v1"):
    manifest = pd.read_csv(root / "data" / dataset / "dataset_manifest.csv")
    if set(manifest["split"]) != {"train", "val", "test"}:
        raise ValueError(f"Incomplete split inventory in {dataset} manifest")
    if manifest[["image_id", "source_path"]].isna().any().any():
        raise ValueError(f"Missing original identity in {dataset} manifest")
    source_ids = manifest["source_path"].map(original_id)
    if not source_ids.equals(manifest["image_id"].map(original_id)):
        raise ValueError(f"Source identity mismatch in {dataset} manifest")
    excluded_ids.update(source_ids)

raw_by_id = {}
for path in raw_test.rglob("*"):
    if path.is_file() and path.suffix.casefold() in {".jpg", ".jpeg", ".png"}:
        image_id = original_id(path.name)
        if image_id in raw_by_id:
            raise ValueError(f"Duplicate original identity in raw test: {image_id}")
        raw_by_id[image_id] = path

eligible_ids = raw_by_id.keys() - excluded_ids
if len(eligible_ids) < sample_size:
    raise ValueError(f"Need {sample_size} eligible test images, found {len(eligible_ids)}")

def sample_rank(image_id: str) -> tuple[str, str]:
    return hashlib.sha256(f"{seed}:{image_id}".encode()).hexdigest(), image_id

sample_ids = sorted(eligible_ids, key=sample_rank)[:sample_size]
sample = pd.DataFrame({
    "image_id": sample_ids,
    "raw_split": "test",
    "source_path": [raw_by_id[key].relative_to(root).as_posix() for key in sample_ids],
})
print(f"{len(raw_by_id)} raw test images, {len(raw_by_id.keys() & excluded_ids)} excluded")
display(sample.head())

## 2. Predict and apply padded black masks

Set `run_inference = True` after review. The run requires MPS and a verified frozen checkpoint. Pick a new output directory for each run. The final directory appears after all predictions and exports succeed.

In [ ]:
def padded_box(box: list[float], width: int, height: int) -> tuple[int, int, int, int]:
    x1, y1, x2, y2 = box
    pad_x = max(minimum_padding_px, padding_fraction * (x2 - x1))
    pad_y = max(minimum_padding_px, padding_fraction * (y2 - y1))
    left = max(0, math.floor(x1 - pad_x))
    top = max(0, math.floor(y1 - pad_y))
    right = min(width, math.ceil(x2 + pad_x))
    bottom = min(height, math.ceil(y2 + pad_y))
    if not (left < right and top < bottom):
        raise ValueError(f"Invalid mask bounds for {box}")
    return left, top, right, bottom

In [ ]:
def export_review(folder: Path, images: pd.DataFrame, detections: pd.DataFrame) -> None:
    fig, axes = plt.subplots(2, 2, figsize=(11, 7), layout="constrained")
    images["detection_count"].value_counts().sort_index().plot.bar(ax=axes[0, 0])
    axes[0, 0].set(xlabel="Detections per image", ylabel="Images")
    for ax, values, label in [
        (axes[0, 1], detections["confidence"], "Detection confidence"),
        (axes[1, 0], detections["box_area_fraction"] * 100, "Box area (% of image)"),
        (axes[1, 1], images["inference_ms"], "Inference (ms per image, batch average)"),
    ]:
        ax.hist(values, bins=25)
        ax.set(xlabel=label, ylabel="Count")
    fig.savefig(folder / "distributions.png", dpi=150)
    plt.close(fig)

    groups = {
        "zero_detections": images.loc[images.detection_count == 0, "image_id"].tolist(),
        "multiple_detections": images.loc[images.detection_count > 1, "image_id"].tolist(),
        "low_confidence": detections.sort_values(["confidence", "image_id"]).image_id.tolist(),
        "small_boxes": detections.sort_values(["box_area_fraction", "image_id"]).image_id.tolist(),
        "random": images.image_id.tolist(),
    }
    review = []
    for reason, ids in groups.items():
        ordered = list(dict.fromkeys(ids))
        if reason in {"zero_detections", "multiple_detections", "random"}:
            ordered.sort(key=sample_rank)
        review.extend({"image_id": key, "reason": reason} for key in ordered[:10])
    pd.DataFrame(review).merge(images, on="image_id").assign(
        reviewed=False, missed_plates="", false_positives="", incomplete_masks="", notes=""
    ).to_csv(folder / "manual_review.csv", index=False)

In [ ]:
def run_batch_inference() -> None:
    if output.exists():
        raise FileExistsError(output)
    if not torch.backends.mps.is_available():
        raise RuntimeError("Apple MPS is unavailable in this notebook kernel")
    if not (frozen / "baseline.json").is_file():
        raise FileNotFoundError("Run the final freeze cell in notebook 05 first")
    baseline = json.loads((frozen / "baseline.json").read_text())
    weights = frozen / "best.pt"
    with weights.open("rb") as stream:
        checksum = hashlib.file_digest(stream, "sha256").hexdigest()
    if checksum != baseline["sha256"]:
        raise RuntimeError("Frozen weights failed checksum verification")

    settings = dict(device="mps", imgsz=baseline["imgsz"], conf=baseline["confidence"],
                    iou=baseline["nms_iou"], batch=batch_size, max_det=300,
                    half=False, rect=False, save=False, verbose=False)
    model = YOLO(str(weights))
    output.parent.mkdir(parents=True, exist_ok=True)
    image_rows, detection_rows = [], []
    box_columns = ["x1", "y1", "x2", "y2"]
    mask_columns = ["mask_left", "mask_top", "mask_right", "mask_bottom"]

    with TemporaryDirectory(dir=output.parent) as staging:
        folder = Path(staging) / output.name
        (folder / "masked").mkdir(parents=True)
        sample.to_csv(folder / "sample.csv", index=False)

        for start in range(0, len(sample), batch_size):
            batch = sample.iloc[start:start + batch_size]
            paths = [str(root / path) for path in batch.source_path]
            started = perf_counter()
            results = model.predict(source=paths, **settings)
            torch.mps.synchronize()
            batch_ms = (perf_counter() - started) * 1000
            if [Path(result.path).resolve() for result in results] != [Path(path).resolve() for path in paths]:
                raise RuntimeError("Prediction results do not match the input batch")

            for row, result in zip(batch.itertuples(index=False), results):
                masked = result.orig_img.copy()
                height, width = masked.shape[:2]
                for index, (box, conf, cls) in enumerate(zip(
                    result.boxes.xyxy.tolist(), result.boxes.conf.tolist(), result.boxes.cls.tolist()
                )):
                    left, top, right, bottom = padded_box(box, width, height)
                    masked[top:bottom, left:right] = 0
                    x1, y1, x2, y2 = box
                    detection_rows.append([
                        row.image_id, index, int(cls), float(conf), *box,
                        left, top, right, bottom, (x2 - x1) * (y2 - y1) / (width * height),
                    ])
                masked_path = f"masked/{row.image_id}.png"
                Image.fromarray(masked[:, :, ::-1]).save(folder / masked_path)
                image_rows.append(dict(
                    image_id=row.image_id, source_path=row.source_path, masked_path=masked_path,
                    width=width, height=height, detection_count=len(result.boxes),
                    preprocess_ms=result.speed["preprocess"], inference_ms=result.speed["inference"],
                    postprocess_ms=result.speed["postprocess"], batch_index=start // batch_size,
                    batch_wall_ms=batch_ms, batch_image_count=len(batch),
                ))
            print(f"Processed {start + len(batch)}/{len(sample)}", end="\r")

        images = pd.DataFrame(image_rows)
        detections = pd.DataFrame(detection_rows, columns=[
            "image_id", "detection_index", "class_id", "confidence",
            *box_columns, *mask_columns, "box_area_fraction",
        ])
        images.to_csv(folder / "images.csv", index=False)
        detections.to_csv(folder / "detections.csv", index=False)
        export_review(folder, images, detections)
        run = dict(baseline=baseline, settings=settings, sampling_seed=seed,
                   padding_fraction=padding_fraction, minimum_padding_px=minimum_padding_px,
                   torch=torch.__version__, ultralytics=ultralytics.__version__,
                   platform=platform.platform(), images=len(images), detections=len(detections),
                   zero_detections=int((images.detection_count == 0).sum()),
                   multiple_detections=int((images.detection_count > 1).sum()))
        (folder / "run.json").write_text(json.dumps(run, indent=2) + "\n")
        folder.rename(output)
    print(f"\nSaved {len(images)} results to {output}")

if run_inference:
    run_batch_inference()
else:
    print("Inference is disabled. Review the sample and settings first.")

## 3. Manual review

Run the next two cells to compare every local source image with its masked PNG. Enter issue counts, add optional notes, and use **Save & next**. Progress is written back to `manual_review.csv` after every save or navigation. Detection and mask coordinates in `detections.csv` are in original pixels; mask end coordinates are exclusive. The review groups are for triage only, and unlabelled predictions alone cannot establish precision or recall.

In [7]:
import ipywidgets as widgets
import pandas as pd

from IPython.display import display
from pathlib import Path

# This cell is restart-safe: it locates the repository and run output itself.
root = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / "notebooks/06_model_v1_large_scale_inference.ipynb").is_file())
output = root / "outputs/model_v1_bdd100k_test"
review_csv = output / "manual_review.csv"
if not review_csv.is_file():
    raise FileNotFoundError(f"Review manifest not found: {review_csv}")

review_df = pd.read_csv(review_csv, keep_default_na=False)
if review_df.empty:
    raise ValueError(f"Review manifest is empty: {review_csv}")
for column in ("missed_plates", "false_positives", "incomplete_masks"):
    review_df[column] = pd.to_numeric(review_df[column], errors="coerce").fillna(0).astype(int)
if "reviewed" not in review_df:
    review_df["reviewed"] = False
else:
    review_df["reviewed"] = review_df["reviewed"].astype(str).str.casefold().eq("true")
review_df["notes"] = review_df["notes"].astype(str)
print(f"Loaded {len(review_df)} review images ({int(review_df['reviewed'].sum())} completed).")

Loaded 50 review images (23 completed).


In [8]:
def resolve_review_path(value: str, base: Path) -> Path:
    path = Path(value)
    resolved = path if path.is_absolute() else base / path
    if not resolved.is_file():
        raise FileNotFoundError(resolved)
    return resolved

def save_reviews() -> None:
    temporary_csv = review_csv.with_suffix(".tmp")
    review_df.to_csv(temporary_csv, index=False)
    temporary_csv.replace(review_csv)

index = widgets.IntSlider(
    value=0, min=0, max=len(review_df) - 1, step=1,
    description="Image", continuous_update=False,
    layout=widgets.Layout(width="70%"),
)
original_image = widgets.Image(layout=widgets.Layout(width="100%", height="auto", object_fit="contain"))
masked_image = widgets.Image(layout=widgets.Layout(width="100%", height="auto", object_fit="contain"))
missed = widgets.BoundedIntText(value=0, min=0, description="Missed")
false_positive = widgets.BoundedIntText(value=0, min=0, description="False +")
incomplete = widgets.BoundedIntText(value=0, min=0, description="Incomplete")
notes = widgets.Textarea(description="Notes", layout=widgets.Layout(width="70%"))
previous_button = widgets.Button(description="Previous")
save_button = widgets.Button(description="Save", button_style="info")
next_button = widgets.Button(description="Save & next", button_style="success")
header = widgets.HTML()
status = widgets.HTML()

def render_review(position: int) -> None:
    row = review_df.iloc[position]
    source_path = resolve_review_path(row["source_path"], root)
    masked_path = resolve_review_path(row["masked_path"], output)
    original_image.value = source_path.read_bytes()
    original_image.format = source_path.suffix.lstrip(".").lower().replace("jpg", "jpeg")
    masked_image.value = masked_path.read_bytes()
    masked_image.format = masked_path.suffix.lstrip(".").lower().replace("jpg", "jpeg")
    missed.value = int(row["missed_plates"])
    false_positive.value = int(row["false_positives"])
    incomplete.value = int(row["incomplete_masks"])
    notes.value = str(row["notes"])
    marker = "✓ reviewed" if bool(row["reviewed"]) else "not reviewed"
    header.value = (f"<b>{position + 1}/{len(review_df)}</b> · {row['image_id']} · "
                    f"{row['reason']} · detections: {row['detection_count']} · {marker}")
    previous_button.disabled = position == 0
    next_button.disabled = position == len(review_df) - 1

def store_review(position: int) -> None:
    review_df.loc[position, ["reviewed", "missed_plates", "false_positives",
                             "incomplete_masks", "notes"]] = [
        True, missed.value, false_positive.value, incomplete.value, notes.value.strip()
    ]
    save_reviews()
    status.value = f"Saved {int(review_df['reviewed'].sum())}/{len(review_df)} reviews."

def change_index(change: dict) -> None:
    if change["name"] == "value":
        store_review(int(change["old"]))
        render_review(int(change["new"]))

def go_previous(_: widgets.Button) -> None:
    index.value -= 1

def save_current(_: widgets.Button) -> None:
    store_review(index.value)
    render_review(index.value)

def save_and_next(_: widgets.Button) -> None:
    store_review(index.value)
    if index.value < index.max:
        index.value += 1

index.observe(change_index, names="value")
previous_button.on_click(go_previous)
save_button.on_click(save_current)
next_button.on_click(save_and_next)

In [9]:
render_review(index.value)
image_panels = [
    widgets.VBox([widgets.HTML("<b>Original</b>"), original_image]),
    widgets.VBox([widgets.HTML("<b>Masked</b>"), masked_image]),
]
previews = widgets.Box(image_panels, layout=widgets.Layout(width="100%"))
full_width = widgets.Checkbox(description="Full-width images", indent=False)

def resize_previews(change: dict) -> None:
    stacked = change["new"]
    previews.layout.flex_direction = "column" if stacked else "row"
    for panel in image_panels:
        panel.layout.width = "100%" if stacked else "50%"
        panel.layout.min_width = "0"

resize_previews({"new": full_width.value})
full_width.observe(resize_previews, names="value")
display(widgets.VBox([
    header,
    full_width,
    previews,
    index,
    widgets.HBox([missed, false_positive, incomplete]),
    notes,
    widgets.HBox([previous_button, save_button, next_button]),
    status,
]))